### Data Ingestion

In [68]:
# Notebook setup: central imports and initializations
import os
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

# Optionally initialize reusable components here (uncomment to run):
# embedding_manager = EmbeddingManager()
# vector_store = VectorStore()

In [69]:
### Document Structure
from langchain_core.documents import Document

In [70]:
doc = Document(
    page_content="This is a sample document.",
    metadata={"source": "sample_source", "author": "John Doe","pages":1,"date_created":"2024-06-01"}
)

In [71]:
## create a simple txt file
import os
os.makedirs("../data/text_files", exist_ok=True)

In [72]:
sample_texts = {
    "../data/text_files/python_intro.txt": """Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular programming languages in the world.

Key Features:
- Easy to learn and use
- Extensive standard library
- Great for AI, web, and automation
""",

    "../data/text_files/ml_intro.txt": """Machine Learning Introduction

Machine Learning is a subset of Artificial Intelligence that enables systems to learn from data.

Key Concepts:
- Supervised Learning
- Unsupervised Learning
- Reinforcement Learning
"""
}

for file_path, content in sample_texts.items():
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(content)
print ("Sample text files created successfully.")

Sample text files created successfully.


In [73]:
### TextLoader
from langchain_community.document_loaders import TextLoader
loader = TextLoader("../data/text_files/python_intro.txt",encoding="utf-8")
document = loader.load()
print(document[0].page_content)

Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular programming languages in the world.

Key Features:
- Easy to learn and use
- Extensive standard library
- Great for AI, web, and automation



In [74]:
 ## Directory Loader
from langchain_community.document_loaders import DirectoryLoader

# Load all text files in the directory
directory_loader = DirectoryLoader("../data/text_files", glob="*.*", loader_cls=TextLoader, loader_kwargs={"encoding": "utf-8"},show_progress=False)
directory_loader.load()

[Document(metadata={'source': '..\\data\\text_files\\ml_intro.txt'}, page_content='Machine Learning Introduction\n\nMachine Learning is a subset of Artificial Intelligence that enables systems to learn from data.\n\nKey Concepts:\n- Supervised Learning\n- Unsupervised Learning\n- Reinforcement Learning\n'),
 Document(metadata={'source': '..\\data\\text_files\\python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular programming languages in the world.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Great for AI, web, and automation\n')]

In [75]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
#load all pdf files in the directory
directory_loader = DirectoryLoader("../data/pdf", glob="*.pdf", loader_cls=PyPDFLoader,show_progress=False)
directory_loader.load()


[Document(metadata={'producer': 'LibreOffice 4.2', 'creator': 'Writer', 'creationdate': '2017-08-16T14:44:13+02:00', 'source': '..\\data\\pdf\\sample-1.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1'}, page_content='Lorem ipsum \nLorem ipsum dolor sit amet, consectetur adipiscing \nelit. Nunc ac faucibus odio. \nVestibulum neque massa, scelerisque sit amet ligula eu, congue molestie mi. Praesent ut\nvarius sem. Nullam at porttitor arcu, nec lacinia nisi. Ut ac dolor vitae odio interdum\ncondimentum.  Vivamus  dapibus  sodales  ex,  vitae  malesuada  ipsum  cursus\nconvallis. Maecenas sed egestas nulla, ac condimentum orci.  Mauris diam felis,\nvulputate ac suscipit et, iaculis non est. Curabitur semper arcu ac ligula semper, nec luctus\nnisl blandit. Integer lacinia ante ac libero lobortis imperdiet. Nullam mollis convallis ipsum,\nac accumsan nunc vehicula vitae. Nulla eget justo in felis tristique fringilla. Morbi sit amet\ntortor quis risus auctor condimentum. Morbi in ullamcor

Embeddings And VectorStore DB

In [76]:
import numpy as np
import os
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [77]:
# Chunking
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """
    Split documents into smaller chunks for better RAG performance.
    
    Parameters:
    - chunk_size: Maximum characters per chunk (adjust based on your LLM)
    - chunk_overlap: Characters to overlap between chunks (preserves context)
    """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, # Each chunk: ~1000 characters
        chunk_overlap=chunk_overlap, # 200 chars overlap for context
        length_function=len, # How to measure length
        separators=["\n\n", "\n", " ", ""] # Split hierarchy
    )
    # Actually split the documents
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show what a chunk looks like
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

chunks = split_documents(directory_loader.load(), chunk_size=1000, chunk_overlap=200)

Split 6 documents into 17 chunks

Example chunk:
Content: Lorem ipsum 
Lorem ipsum dolor sit amet, consectetur adipiscing 
elit. Nunc ac faucibus odio. 
Vestibulum neque massa, scelerisque sit amet ligula eu, congue molestie mi. Praesent ut
varius sem. Nulla...
Metadata: {'producer': 'LibreOffice 4.2', 'creator': 'Writer', 'creationdate': '2017-08-16T14:44:13+02:00', 'source': '..\\data\\pdf\\sample-1.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1'}


In [78]:
from sentence_transformers import SentenceTransformer
class EmbeddingManager:
    """Handles document embeddings generation using SentenceTransformer."""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name : Hugging Face model name for sentence embeddings.
        """
        self.model_name = model_name
        self.model = None
        self._load_model()
        
    def _load_model(self):
        """Load the SentenceTransformer model."""
        try:
            print(f"Loading model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model Loaded successfully. Embedding Dimensions: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model: {e}")
    
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts.
        
        Args:
            texts : List of text strings to embed.
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dimension).
            
        """
        
        if not self.model:
            raise ValueError("Model is not loaded. Please initialize the model first.")
        
        embeddings = self.model.encode(texts, convert_to_numpy=True)
        print(f"Generated embeddings for {len(texts)} texts. Shape: {embeddings.shape}")
        return embeddings
    
embedding_manager = EmbeddingManager()
embedding_manager

Loading model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5155.54it/s]


Model Loaded successfully. Embedding Dimensions: 384


### VectorStore

In [79]:
class VectorStore:
    """Manages document embeddings and their storage in a ChromaDB collection."""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store.
        
        Args:
            collection_name : Name of the ChromaDB collection.
            persist_directory : Directory to persist the vector store data.
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()
        
    def _initialize_store(self):
        """Initialize the ChromaDB client and collection."""
        try:
            # create persist directory if not exists
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # get or create collection
            self.collection = self.client.get_or_create_collection(name=self.collection_name, metadata={"hnsw:space": "cosine", "description": "PDF document embeddings for RAG"})
            print(f"Vector store initialized. Collection: {self.collection_name}, Persist Directory: {self.persist_directory}")
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            
    
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings in vectorstore.
        
        Args:
            documents : List of Langchain Documents.
            embeddings : Array of corresponding embeddings.
        """
        
        if len(documents) != len(embeddings):
            raise ValueError("The number of documents and embeddings must match.")
        
        print(f"Adding {len(documents)} documents to the vector store...")
        
        # prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_texts = []
        embeddings_list = []
        
        for i, (doc,embedding) in enumerate(zip(documents, embeddings)):
            
            # generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # prepare metadata
            metadata = dict(doc.metadata)  # Ensure it's a dictionary
            metadata["document_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)
            
            # document content
            documents_texts.append(doc.page_content)
            
            # embedding
            embeddings_list.append(embedding.tolist())
            
        # Add to collection
                   
        try:
            
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_texts
            )
            print(f"Successfully added {len(documents)} documents to the vector store.")
            print(f"Collection now contains {self.collection.count()} documents.")
            
        except Exception as e:
            print(f"Error adding documents: {e}")
            
vector_store = VectorStore()
vector_store


Vector store initialized. Collection: pdf_documents, Persist Directory: ../data/vector_store


In [80]:
chunks

[Document(metadata={'producer': 'LibreOffice 4.2', 'creator': 'Writer', 'creationdate': '2017-08-16T14:44:13+02:00', 'source': '..\\data\\pdf\\sample-1.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1'}, page_content='Lorem ipsum \nLorem ipsum dolor sit amet, consectetur adipiscing \nelit. Nunc ac faucibus odio. \nVestibulum neque massa, scelerisque sit amet ligula eu, congue molestie mi. Praesent ut\nvarius sem. Nullam at porttitor arcu, nec lacinia nisi. Ut ac dolor vitae odio interdum\ncondimentum.  Vivamus  dapibus  sodales  ex,  vitae  malesuada  ipsum  cursus\nconvallis. Maecenas sed egestas nulla, ac condimentum orci.  Mauris diam felis,\nvulputate ac suscipit et, iaculis non est. Curabitur semper arcu ac ligula semper, nec luctus\nnisl blandit. Integer lacinia ante ac libero lobortis imperdiet. Nullam mollis convallis ipsum,\nac accumsan nunc vehicula vitae. Nulla eget justo in felis tristique fringilla. Morbi sit amet\ntortor quis risus auctor condimentum. Morbi in ullamcor

In [81]:
# convert the text into embeddings
texts = [doc.page_content for doc in chunks]

# generate the embeddings
embeddings = embedding_manager.generate_embeddings(texts)

# store the documents and embeddings in the vector store
vectors = vector_store.add_documents(chunks, embeddings)
vectors

Generated embeddings for 17 texts. Shape: (17, 384)
Adding 17 documents to the vector store...
Successfully added 17 documents to the vector store.
Collection now contains 91 documents.


Retriever Pipeline From VectorStore

In [82]:
class RAGRetriever:
    """Retrieves relevant documents from the vector store based on a query."""
    
    def __init__(self, vector_store: VectorStore,embedding_manager: EmbeddingManager):
        """
        Initialize the RAG retriever.
        
        Args:
            vector_store : An instance of the VectorStore class.
            embedding_manager : An instance of the EmbeddingManager class.
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
        
    def retrieve(self, query: str, top_k: int = 5,score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve top_k relevant documents for a given query.
        
        Args:
            query : The input query string.
            top_k : Number of top documents to retrieve.
            score_threshold : Minimum similarity score for retrieved documents.
            
        Returns:
            List of dictionaries containing document content and metadata.
        """
        
        print(f"Retreiving documents for query: '{query}' with top_k={top_k} and score_threshold={score_threshold}")
        
        # Generate embedding for the query
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k,
                )
            
            # process results
            retrieved_docs = []
            
            if results["documents"] and results["documents"][0]:
                documents = results["documents"][0]
                metadatas = results["metadatas"][0]
                distances = results["distances"][0]
                ids = results["ids"][0]
                
                for i,(doc_id,document,metadate,distance) in enumerate(zip(ids,documents,metadatas,distances)):
                    # convert distance to similarity score (chromadb uses cosine disance)
                    
                    similarity_score = 1 - distance  # Convert distance to similarity score
                    if(similarity_score >= score_threshold):
                       retrieved_docs.append({
                            "id": doc_id,
                            "content": document,
                            "metadata": metadate,
                            "similarity_score": similarity_score,
                            "distance": distance,
                            "rank": i + 1
                       })
                       
                print(f"Retrieved {len(retrieved_docs)} documents for query: '{query}'")
                
            else:
                print(f"No documents found for query: '{query}'")
        
            return retrieved_docs
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []
        

rag_retriever = RAGRetriever(vector_store=vector_store, embedding_manager=embedding_manager)
rag_retriever

In [83]:
rag_retriever.retrieve("sample pdf?")

Retreiving documents for query: 'sample pdf?' with top_k=5 and score_threshold=0.0
Generated embeddings for 1 texts. Shape: (1, 384)
Retrieved 5 documents for query: 'sample pdf?'


[{'id': 'doc_778f061e_9',
  'content': 'Sample PDFThis is a simple PDF ﬁle. Fun fun fun.\nLorem ipsum dolor sit amet, consectetuer adipiscing elit. Phasellus facilisis odio sed mi. \nCurabitur suscipit. Nullam vel nisi. Etiam semper ipsum ut lectus. Proin aliquam, erat eget \npharetra commodo, eros mi condimentum quam, sed commodo justo quam ut velit. \nInteger a erat. Cras laoreet ligula cursus enim. Aenean scelerisque velit et tellus. \nVestibulum dictum aliquet sem. Nulla facilisi. Vestibulum accumsan ante vitae elit. Nulla \nerat dolor, blandit in, rutrum quis, semper pulvinar, enim. Nullam varius congue risus. \nVivamus sollicitudin, metus ut interdum eleifend, nisi tellus pellentesque elit, tristique \naccumsan eros quam et risus. Suspendisse libero odio, mattis sit amet, aliquet eget, \nhendrerit vel, nulla. Sed vitae augue. Aliquam erat volutpat. Aliquam feugiat vulputate nisl. \nSuspendisse quis nulla pretium ante pretium mollis. Proin velit ligula, sagittis at, egestas a, \np

In [84]:
rag_retriever.retrieve("Maecenas mauris lectus, lobortis et purus mattis, blandit dictum tellus.")

Retreiving documents for query: 'Maecenas mauris lectus, lobortis et purus mattis, blandit dictum tellus.' with top_k=5 and score_threshold=0.0
Generated embeddings for 1 texts. Shape: (1, 384)
Retrieved 5 documents for query: 'Maecenas mauris lectus, lobortis et purus mattis, blandit dictum tellus.'


[{'id': 'doc_67f95fdb_1',
  'content': 'mauris tempus fringilla.\nMaecenas mauris lectus, lobortis et purus mattis, blandit dictum tellus.\n\uf0b7 Maecenas non lorem quis tellus placerat varius. \n\uf0b7 Nulla facilisi. \n\uf0b7 Aenean congue fringilla justo ut aliquam. \n\uf0b7 Mauris id ex erat. Nunc vulputate neque vitae justo facilisis, non condimentum ante\nsagittis. \n\uf0b7 Morbi viverra semper lorem nec molestie. \n\uf0b7 Maecenas tincidunt est efficitur ligula euismod, sit amet ornare est vulputate.\nRow 1 Row 2 Row 3 Row 4\n0\n2\n4\n6\n8\n10\n12\nColumn 1\nColumn 2\nColumn 3',
  'metadata': {'source': 'data/pdf/sample-1.pdf',
   'page': 0,
   'page_label': '1',
   'creationdate': '2017-08-16T14:44:13+02:00',
   'producer': 'LibreOffice 4.2',
   'creator': 'Writer',
   'total_pages': 4},
  'similarity_score': 0.6862208843231201,
  'distance': 0.3137791156768799,
  'rank': 1},
 {'id': 'doc_593148c4_1',
  'content': 'mauris tempus fringilla.\nMaecenas mauris lectus, lobortis et 

In [85]:
from dotenv import load_dotenv
from pathlib import Path
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.messages import HumanMessage
# from langchain.schema import HumanMessage

# Load API key from .env at the repository root
load_dotenv(dotenv_path=Path.cwd() / '.env')
google_api_key = os.getenv('GOOGLE_API_KEY')
if not google_api_key:
    raise ValueError('Missing GOOGLE_API_KEY in .env. Add it to the repo root and restart the notebook.')

llm = ChatGoogleGenerativeAI(api_key=google_api_key, model='gemini-3.5-flash')
print('Gemini LLM initialized.')

# simple Rag Function: retrieve context: generate resp

def rag_simple(query: str,retriever: RAGRetriever, llm: ChatGoogleGenerativeAI, top_k: int = 5, score_threshold: float = 0.0) -> str:
    results = retriever.retrieve(query, top_k=top_k, score_threshold=score_threshold)
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found."

    #generate response using the LLM with the retrieved context
    prompt = f""" Use the following context to answer the question concisely.
    Context:
    {context}
    Question:
    {query}"
    
    Answer:"""
    response = llm.invoke(prompt)
    return response.content


Gemini LLM initialized.


In [89]:
answer = rag_simple("What are the skills Waqas Ahmed should improve, even if it is not explicitly mentioned?", rag_retriever, llm, top_k=3, score_threshold=0.0)
answer

Retreiving documents for query: 'What are the skills Waqas Ahmed should improve, even if it is not explicitly mentioned?' with top_k=3 and score_threshold=0.0
Generated embeddings for 1 texts. Shape: (1, 384)
Retrieved 3 documents for query: 'What are the skills Waqas Ahmed should improve, even if it is not explicitly mentioned?'


[{'type': 'text',
  'text': 'Based on his profile and skills, Waqas Ahmed could improve in the following areas:\n\n1. **Modern Frontend Frameworks:** While he has experience with ASP.NET MVC and Razor Pages, adding modern frontend frameworks (like React, Angular, or Vue.js) would make him a more versatile full-stack developer.\n2. **Automated Testing:** Despite emphasizing clean architecture, SOLID principles, and code quality, he does not explicitly list testing methodologies (such as Unit Testing, Integration Testing, or TDD).\n3. **NoSQL Databases:** His database skills are focused solely on SQL Server; learning NoSQL databases (like MongoDB or Redis) would diversify his data management skills.',
  'extras': {'signature': 'ErMmCrAmARFNMg+40xXkt6VId/QpciTw0OEv25o8bKQmPJ57UBpUHxLNv2kllsv8cRYcGLhhgG7fyed8juszOLEcZfIF1eGJkDlIPU1dlF1AmSiK6aB5dD2MVi6mDcmCpmPIp2CgQQFO1bOTH3+ARSMyzGZyRdlDbsMl5csxnYpVvSKmN0uKJZoop43JO/1zQS5tCGySjYP9IqQCV6EbIy493ELdkNTpAtrqSL7Z34Qz/Z3lnaOXbZ71b+KsbN7UbjtctITk